# Анализатор отзывов

Домашнее задание по курсу Python (модули 1–2).

Программа принимает список отзывов покупателей с оценками, обрабатывает их и выводит сводный отчет: какие слова встречаются чаще всего, сколько уникальных слов использовано, какова средняя оценка и как распределились отзывы по настроению.

**Как запустить:** «Среда выполнения» → «Перезапустить и выполнить все». Последняя ячейка запускает интерактивное меню и ждет ввода — для завершения введите `0`.

### Выбор структур данных

| Структура | Где используется | Почему именно она |
|---|---|---|
| кортеж | отзыв `(текст, оценка)`, пара `(слово, количество)` в топе | пара не должна меняться после создания |
| список | набор отзывов, оценки, топ слов, строки отчета | упорядоченная последовательность, порядок важен |
| словарь | частота слов, статистика оценок | быстрый доступ к значению по ключу |
| множество | уникальные слова | повторы отбрасываются автоматически |

## Исходные данные

Каждый отзыв — кортеж из текста и оценки от 1 до 5. Все отзывы хранятся в списке.

In [1]:
# Каждый отзыв — кортеж (текст, оценка): пара не должна меняться после создания.
reviews = [
    ("Отличный товар быстрая доставка", 5),
    ("Товар хороший но доставка медленная", 4),
    ("Ужасный товар не советую никому", 1),
    ("Доставка быстрая товар нормальный", 4),
    ("Отличный магазин отличный сервис", 5),
    ("Плохой товар сломался сразу", 2),
]

Для проверок в каждой части используется небольшая функция `check`: она печатает результат и сравнивает его с ожидаемым. Если результат неверный, ячейка остановится с ошибкой.

In [2]:
def check(actual, expected):
    """Печатает результат и проверяет, что он совпадает с ожидаемым."""
    assert actual == expected, f"ожидалось {expected!r}, получено {actual!r}"
    print("OK:", repr(actual))

## Часть 1. Нормализация текста

Функция `normalize(text)` приводит текст к единому виду. Методы строк вызываются цепочкой в порядке из условия: сначала `.lower()`, затем `.strip()`, затем `.replace()` для запятых и точек. Поэтому точка в конце строки превращается в пробел уже после обрезки краев — как в первом примере из условия.

Если на вход подана не строка, функция возвращает пустую строку.

In [3]:
def normalize(text):
    """Приводит текст к нижнему регистру, убирает пробелы по краям
    и заменяет запятые и точки на пробелы. Для не-строки возвращает ""."""
    if not isinstance(text, str):
        return ""
    return text.lower().strip().replace(",", " ").replace(".", " ")

In [4]:
check(normalize("Отличный, товар."), "отличный  товар ")
check(normalize("  БЫСТРАЯ доставка  "), "быстрая доставка")
check(normalize("Товар, хороший. Очень!"), "товар  хороший  очень!")

# Особый случай: на вход подана не строка
check(normalize(None), "")
check(normalize(123), "")

OK: 'отличный  товар '
OK: 'быстрая доставка'
OK: 'товар  хороший  очень!'
OK: ''
OK: ''


## Часть 2. Подсчет частоты слов

`word_frequency(reviews)` перебирает отзывы циклом `for`. Из каждого кортежа нужен только текст, поэтому оценка распаковывается в переменную `_`. Текст нормализуется функцией из части 1 и разбивается на слова методом `.split()`, а счетчик накапливается в словаре: `.get(word, 0)` возвращает 0 для слова, которое встретилось впервые.

`.split()` без аргументов делит строку по любому количеству пробелов, поэтому двойные пробелы, оставшиеся после замены запятых, не дают пустых «слов».

In [5]:
def word_frequency(reviews):
    """Возвращает словарь {слово: сколько раз встретилось во всех отзывах}."""
    frequency = {}
    for text, _ in reviews:
        for word in normalize(text).split():
            frequency[word] = frequency.get(word, 0) + 1
    return frequency

In [6]:
data = [("Товар хороший", 5), ("Товар плохой", 1)]
check(word_frequency(data), {"товар": 2, "хороший": 1, "плохой": 1})

# Регистр и знаки препинания не влияют на подсчет
check(word_frequency([("Товар,товар. ТОВАР", 5)]), {"товар": 3})

# Особый случай: отзывов нет
check(word_frequency([]), {})

print()
print("Частота слов в исходных отзывах:")
print(word_frequency(reviews))

OK: {'товар': 2, 'хороший': 1, 'плохой': 1}
OK: {'товар': 3}
OK: {}

Частота слов в исходных отзывах:
{'отличный': 3, 'товар': 5, 'быстрая': 2, 'доставка': 3, 'хороший': 1, 'но': 1, 'медленная': 1, 'ужасный': 1, 'не': 1, 'советую': 1, 'никому': 1, 'нормальный': 1, 'магазин': 1, 'сервис': 1, 'плохой': 1, 'сломался': 1, 'сразу': 1}


## Часть 3. Словарь уникальных слов

`unique_words(reviews)` добавляет каждое слово в множество методом `.add()`. Множество само отбрасывает повторы: сколько бы раз слово ни встретилось, оно хранится один раз. Общее число уникальных слов — `len(unique_words(reviews))`.

In [7]:
def unique_words(reviews):
    """Возвращает множество всех различных слов из отзывов.
    Их количество — len(unique_words(reviews))."""
    words = set()
    for text, _ in reviews:
        for word in normalize(text).split():
            words.add(word)
    return words

In [8]:
data = [("товар хороший товар", 5), ("товар плохой", 1)]
check(unique_words(data), {"товар", "хороший", "плохой"})
check(len(unique_words(data)), 3)

# Особый случай: отзывов нет
check(unique_words([]), set())

print()
print("Уникальных слов в исходных отзывах:", len(unique_words(reviews)))

OK: {'плохой', 'хороший', 'товар'}
OK: 3
OK: set()

Уникальных слов в исходных отзывах: 17


## Часть 4. Топ самых частых слов

`top_words(reviews, n)` берет словарь частот из части 2, сортирует его пары `.items()` функцией `sorted()` по второму элементу кортежа (`key`) в порядке убывания (`reverse=True`) и возвращает срез из первых `n` элементов.

- Если `n` больше числа слов, срез просто вернет все слова.
- При `n <= 0` возвращается пустой список: иначе отрицательный срез, например `[:-1]`, отрезал бы слова с конца.
- Сортировка в Python устойчивая: слова с одинаковой частотой остаются в том порядке, в котором впервые встретились в отзывах.

In [9]:
def top_words(reviews, n):
    """Возвращает список из n пар (слово, количество) по убыванию частоты.
    Если слов меньше n — возвращает все; при n <= 0 — пустой список."""
    if n <= 0:
        return []
    frequency = word_frequency(reviews)
    ranked = sorted(frequency.items(), key=lambda pair: pair[1], reverse=True)
    return ranked[:n]

In [10]:
data = [("товар товар товар", 5), ("доставка доставка", 4), ("сервис", 5)]
check(top_words(data, 2), [("товар", 3), ("доставка", 2)])

# Особые случаи: n больше числа слов, n не положительное, отзывов нет
check(top_words(data, 100), [("товар", 3), ("доставка", 2), ("сервис", 1)])
check(top_words(data, 0), [])
check(top_words(data, -1), [])
check(top_words([], 3), [])

OK: [('товар', 3), ('доставка', 2)]
OK: [('товар', 3), ('доставка', 2), ('сервис', 1)]
OK: []
OK: []
OK: []


## Часть 5. Анализ оценок

`rating_stats(reviews)` собирает оценки в отдельный список и считает среднюю, максимальную и минимальную оценку встроенными функциями `sum()`, `len()`, `max()` и `min()`. Затем оценки перебираются циклом и распределяются по настроению: 4–5 — положительные, 3 — нейтральные, 1–2 — отрицательные.

- В список попадают только целые числа от 1 до 5 (условия объединены оператором `and`), поэтому сумма трех категорий всегда равна числу учтенных оценок.
- Для пустого списка функция возвращает нули по всем ключам. Без этой проверки было бы деление на ноль, а `max()` и `min()` завершились бы ошибкой.

In [11]:
def rating_stats(reviews):
    """Возвращает среднюю, максимальную и минимальную оценку и число
    положительных (4–5), нейтральных (3) и отрицательных (1–2) отзывов.
    Оценки вне диапазона 1–5 не учитываются."""
    stats = {"average": 0, "max": 0, "min": 0,
             "positive": 0, "negative": 0, "neutral": 0}

    ratings = []
    for _, rating in reviews:
        if isinstance(rating, int) and 1 <= rating <= 5:
            ratings.append(rating)

    if not ratings:
        return stats

    stats["average"] = sum(ratings) / len(ratings)
    stats["max"] = max(ratings)
    stats["min"] = min(ratings)
    for rating in ratings:
        if rating >= 4:
            stats["positive"] += 1
        elif rating == 3:
            stats["neutral"] += 1
        else:
            stats["negative"] += 1
    return stats

In [12]:
data = [("текст", 5), ("текст", 4), ("текст", 1), ("текст", 3)]
check(rating_stats(data), {"average": 3.25, "max": 5, "min": 1,
                           "positive": 2, "negative": 1, "neutral": 1})

# Особый случай: отзывов нет
check(rating_stats([]), {"average": 0, "max": 0, "min": 0,
                         "positive": 0, "negative": 0, "neutral": 0})

# Особый случай: оценки вне диапазона 1–5 не учитываются
check(rating_stats([("текст", 5), ("текст", 7), ("текст", "5"), ("текст", 2)]),
      {"average": 3.5, "max": 5, "min": 2,
       "positive": 1, "negative": 1, "neutral": 0})

OK: {'average': 3.25, 'max': 5, 'min': 1, 'positive': 2, 'negative': 1, 'neutral': 1}
OK: {'average': 0, 'max': 0, 'min': 0, 'positive': 0, 'negative': 0, 'neutral': 0}
OK: {'average': 3.5, 'max': 5, 'min': 2, 'positive': 1, 'negative': 1, 'neutral': 0}


## Часть 6. Формирование отчета

`make_report(reviews)` объединяет функции из частей 2–5 и собирает отчет из списка строк, соединяя их методом `"\n".join()`. Значения подставляются f-строками, средняя оценка выводится с двумя знаками после запятой (`:.2f`). Функция возвращает строку, а печатает ее вызывающий код.

Вспомогательные функции `format_top`, `format_mood` и `format_stats` оформляют повторяющиеся фрагменты вывода. Они используются и в отчете, и в меню из части 7, поэтому код оформления не дублируется.

In [13]:
TOP_COUNT = 3  # сколько слов показывать в топе по умолчанию

def format_top(pairs):
    """Превращает пары (слово, количество) в строки вида «  слово — 3»."""
    if not pairs:
        return ["  (слов нет)"]
    lines = []
    for word, count in pairs:
        lines.append(f"  {word} — {count}")
    return lines

def format_mood(stats):
    """Строки с распределением отзывов по настроению."""
    return [
        f"Положительных: {stats['positive']}",
        f"Нейтральных: {stats['neutral']}",
        f"Отрицательных: {stats['negative']}",
    ]

def format_stats(stats):
    """Строки с полной статистикой оценок."""
    lines = [
        f"Средняя оценка: {stats['average']:.2f}",
        f"Максимальная оценка: {stats['max']}",
        f"Минимальная оценка: {stats['min']}",
    ]
    return lines + format_mood(stats)

def make_report(reviews):
    """Собирает сводный отчет по отзывам и возвращает его строкой."""
    stats = rating_stats(reviews)
    lines = [
        "ОТЧЕТ ПО ОТЗЫВАМ",
        f"Всего отзывов: {len(reviews)}",
        f"Уникальных слов: {len(unique_words(reviews))}",
        f"Средняя оценка: {stats['average']:.2f}",
        "",
    ]
    lines += format_mood(stats)
    lines += ["", f"Топ-{TOP_COUNT} слова:"]
    lines += format_top(top_words(reviews, TOP_COUNT))
    return "\n".join(lines)

In [14]:
print(make_report(reviews))

ОТЧЕТ ПО ОТЗЫВАМ
Всего отзывов: 6
Уникальных слов: 17
Средняя оценка: 3.50

Положительных: 4
Нейтральных: 0
Отрицательных: 2

Топ-3 слова:
  товар — 5
  отличный — 3
  доставка — 3


> **Примечание к примеру из условия.** Пример отчета в условии показывает формат, но его числа не совпадают с исходными данными. Проверка вручную по шести отзывам:
>
> - «товар» есть в отзывах 1, 2, 3, 4 и 6 — **5** раз;
> - «отличный» — один раз в отзыве 1 и дважды в отзыве 5 — **3** раза;
> - «доставка» — в отзывах 1, 2 и 4 — **3** раза;
> - положительных оценок (4 или 5) — **4**: 5, 4, 4, 5;
> - уникальных слов — **17**.
>
> Отчет выше совпадает с ручным подсчетом. Ячейка ниже проверяет эти числа автоматически.

In [15]:
check(len(reviews), 6)
check(len(unique_words(reviews)), 17)
check(rating_stats(reviews)["positive"], 4)
check(top_words(reviews, 3), [("товар", 5), ("отличный", 3), ("доставка", 3)])

# Особый случай: отзывов нет — отчет строится без ошибок
print()
print(make_report([]))

OK: 6
OK: 17
OK: 4
OK: [('товар', 5), ('отличный', 3), ('доставка', 3)]

ОТЧЕТ ПО ОТЗЫВАМ
Всего отзывов: 0
Уникальных слов: 0
Средняя оценка: 0.00

Положительных: 0
Нейтральных: 0
Отрицательных: 0

Топ-3 слова:
  (слов нет)


## Часть 7. Интерактивное меню

`run()` в цикле `while True` показывает меню и выполняет выбранное действие. Ввод пользователя — строка, поэтому он сравнивается со строками `"1"`, `"2"`, `"3"` и `"0"`. На любой другой ввод выводится подсказка, и меню показывается снова. Цикл завершается только оператором `break` при выборе `0`.

В пункте «Топ слов» программа спрашивает, сколько слов показать (Enter — 3). Если введено не целое положительное число, выводится сообщение, и программа возвращается в меню. Если запрошено больше слов, чем есть, показываются все.

По умолчанию меню работает с исходным списком `reviews`, но ему можно передать и другой список. Например, `run([])` тоже работает без ошибок.

In [16]:
MENU = """
1 — Полный отчет
2 — Топ слов
3 — Статистика оценок
0 — Выход"""

def ask_top_count():
    """Спрашивает, сколько слов показать. При неверном вводе возвращает None."""
    answer = input(f"Сколько слов показать? (Enter — {TOP_COUNT}): ").strip()
    if answer == "":
        return TOP_COUNT
    if answer.isdecimal() and int(answer) > 0:
        return int(answer)
    print("Нужно ввести целое число больше нуля.")
    return None

def show_top_words(data):
    """Выводит топ слов в количестве, которое выбрал пользователь."""
    count = ask_top_count()
    if count is None:
        return
    pairs = top_words(data, count)
    if 0 < len(pairs) < count:
        print(f"Уникальных слов всего {len(pairs)}, показаны все.")
    print("\n".join(format_top(pairs)))

def run(data=reviews):
    """Показывает меню и выполняет выбранные действия, пока не выбран выход."""
    while True:
        print(MENU)
        choice = input("Выберите действие: ").strip()
        if choice == "1":
            print(make_report(data))
        elif choice == "2":
            show_top_words(data)
        elif choice == "3":
            print("\n".join(format_stats(rating_stats(data))))
        elif choice == "0":
            print("До свидания!")
            break
        else:
            print("Неизвестная команда, введите 1, 2, 3 или 0.")

## Запуск программы

Ячейка ниже ждет ввода: введите номер пункта и нажмите Enter. Для выхода введите `0`.

In [ ]:
run()